# DIMMS and RINGSS Nightly Summary

Author(s): Bruno Quint
Last Update: 2025-10-24

# Table of Contents


In [ ]:
day_obs = 20251020

In [ ]:
# Notebook Setup
import os    
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy.time import Time
from sklearn.cluster import DBSCAN
from datetime import timedelta

from lsst.ts.xml.enums.DIMM import ScopeMotion
from lsst.summit.utils.efdUtils import getDayObsStartTime, getDayObsEndTime, getEfdData, makeEfdClient

In [ ]:
efd_client = makeEfdClient()

start_time = getDayObsStartTime(day_obs)
end_time = getDayObsEndTime(day_obs)

topics_and_columns = {
    "lsst.sal.DIMM.status": ["ra", "decl", "azimuth", "altitude", "motionState", "salIndex"]
}

In [ ]:
dimm_key = "lsst.sal.DIMM.status"
dimm_data = getEfdData(
    client=efd_client, 
    topic=dimm_key,
    columns=topics_and_columns[dimm_key],
    begin=start_time,
    end=end_time
)

# Our data has lots of NaN's. Let's drop them for now.
dimm_data = dimm_data.dropna(how="all")

# Clear out rows where ra/dec are both 0.
dimm_data = dimm_data[dimm_data["ra"] != 0]
dimm_data = dimm_data[dimm_data["decl"] != 0]

# Map the motion state so it is human readable
dimm_data["motionStateName"] = dimm_data["motionState"].map(lambda x: ScopeMotion(x).name)

# Maybe save the data
# dimm_data.to_csv("dimm_data.csv")

# Tower DIMM and Portable DIMM comparisons

Let's start makind sure that we are comparing apples to apples.  
By the time we wrote this notebook, the `motionState` had unreliable data.  
The analysis will probably be easier when we have it.  

For this analysis, we will use [sklearn.cluster.DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html).

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Clustering parameters (optimized for noisy data)
CLUSTERING_EPS = 0.1  # degrees - spatial tolerance for grouping observations
CLUSTERING_MIN_SAMPLES = 20  # minimum observations to form a cluster (increased from 5)

# Quality filtering thresholds
MIN_OBSERVATIONS = 50  # minimum observations to be considered a significant target
MIN_DURATION_MINUTES = 10  # OR minimum duration in minutes

# Shared target matching tolerances
SHARED_TARGET_RA_TOLERANCE = 0.15  # degrees
SHARED_TARGET_DECL_TOLERANCE = 0.15  # degrees

# Temporal overlap detection
SIMULTANEOUS_TIME_WINDOW_MINUTES = 5  # time window for "simultaneous" observations

In [ ]:
print("="*80)
print("DIMM MULTI-HARDWARE CLUSTERING ANALYSIS v2.0")
print("="*80)
print("\nLOADING DATA")
print("-"*80)

df = dimm_data

print(f"Total records: {len(df):,}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nRecords by hardware:")
print(df['salIndex'].value_counts().sort_index())

# Map salIndex to hardware names
hardware_names = {1: 'Tower DIMM', 2: 'Portable DIMM'}
df['hardware'] = df['salIndex'].map(hardware_names)

# Filter valid data (non-null RA/DECL)
tracking_data = df[df['ra'].notna() & df['decl'].notna()].copy()

print(f"\nValid tracking records: {len(tracking_data):,}")
print(f"  Tower DIMM (salIndex=1): {len(tracking_data[tracking_data['salIndex']==1]):,}")
print(f"  Portable DIMM (salIndex=2): {len(tracking_data[tracking_data['salIndex']==2]):,}")

In [ ]:
# ============================================================================
# STEP 2: Cluster targets for each DIMM separately
# ============================================================================

print("\n" + "="*80)
print("CLUSTERING TARGETS BY HARDWARE")
print("="*80)
print(f"\nParameters: eps={CLUSTERING_EPS}°, min_samples={CLUSTERING_MIN_SAMPLES}")
print(f"Quality filters: ≥{MIN_OBSERVATIONS} obs OR ≥{MIN_DURATION_MINUTES} min duration")


def cluster_targets(data, hardware_name, eps=CLUSTERING_EPS, min_samples=CLUSTERING_MIN_SAMPLES):
    """
    Cluster targets using DBSCAN algorithm with robust parameters
    
    Parameters:
    -----------
    data : DataFrame
        Tracking data for a single hardware
    hardware_name : str
        Name of the hardware (for display)
    eps : float
        Maximum distance between points in same cluster (degrees)
    min_samples : int
        Minimum points to form a cluster
    
    Returns:
    --------
    DataFrame with target_id column added, dict with statistics
    """
    print(f"\n{hardware_name}:")
    print(f"  Records to cluster: {len(data):,}")
    
    # Prepare coordinates
    coords = data[['ra', 'decl']].values
    
    # Perform clustering
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    
    # Add cluster labels
    data_clustered = data.copy()
    data_clustered['target_id'] = clustering.labels_
    
    # Initial statistics
    n_clusters_initial = len(set(clustering.labels_)) - (1 if -1 in clustering.labels_ else 0)
    n_noise_initial = sum(clustering.labels_ == -1)
    
    print(f"  Initial clusters identified: {n_clusters_initial}")
    print(f"  Initial noise points: {n_noise_initial}")
    
    # Calculate cluster statistics for filtering
    valid_data = data_clustered[data_clustered['target_id'] >= 0]
    
    if len(valid_data) > 0:
        cluster_stats = valid_data.groupby('target_id').agg({
            'ra': 'count'
        })
        cluster_stats.columns = ['count']
        
        # Calculate durations
        time_info = valid_data.groupby('target_id').apply(
            lambda x: (x.index.max() - x.index.min()).total_seconds() / 60,
            include_groups=False
        )
        cluster_stats['duration_min'] = time_info
        
        # Apply quality filters
        significant_mask = (
            (cluster_stats['count'] >= MIN_OBSERVATIONS) | 
            (cluster_stats['duration_min'] >= MIN_DURATION_MINUTES)
        )
        significant_cluster_ids = cluster_stats[significant_mask].index.tolist()
        
        # Filter the data to keep only significant clusters
        # Mark filtered clusters as noise (-1)
        data_clustered.loc[
            (data_clustered['target_id'] >= 0) & 
            (~data_clustered['target_id'].isin(significant_cluster_ids)),
            'target_id'
        ] = -1
        
        # Renumber clusters sequentially starting from 0
        old_to_new = {old_id: new_id for new_id, old_id in enumerate(sorted(significant_cluster_ids))}
        data_clustered.loc[data_clustered['target_id'] >= 0, 'target_id'] = \
            data_clustered.loc[data_clustered['target_id'] >= 0, 'target_id'].map(old_to_new)
        
        n_clusters_final = len(significant_cluster_ids)
        n_noise_final = sum(data_clustered['target_id'] == -1)
        n_filtered = n_clusters_initial - n_clusters_final
        
        print(f"  Clusters after filtering: {n_clusters_final}")
        print(f"  Filtered out: {n_filtered} small/short clusters")
        print(f"  Final noise/filtered points: {n_noise_final}")
        
        # Statistics for filtered clusters
        small_clusters = cluster_stats[~significant_mask]
        if len(small_clusters) > 0:
            print(f"  Filtered cluster sizes: {small_clusters['count'].min():.0f} to {small_clusters['count'].max():.0f} obs")
            print(f"  Filtered observations: {small_clusters['count'].sum():.0f}")
    
    stats = {
        'initial_clusters': n_clusters_initial,
        'final_clusters': n_clusters_final if len(valid_data) > 0 else 0,
        'filtered_clusters': n_filtered if len(valid_data) > 0 else 0,
        'noise_points': n_noise_final if len(valid_data) > 0 else n_noise_initial
    }
    
    return data_clustered, stats


# Cluster each hardware separately
tower_data, tower_stats = cluster_targets(
    tracking_data[tracking_data['salIndex'] == 1],
    'Tower DIMM (salIndex=1)'
)

portable_data, portable_stats = cluster_targets(
    tracking_data[tracking_data['salIndex'] == 2],
    'Portable DIMM (salIndex=2)'
)

# Combine back into single dataframe with hardware-specific target IDs
tower_data['global_target_id'] = tower_data['target_id'].apply(
    lambda x: f"T{x}" if x >= 0 else "T-1"
)
portable_data['global_target_id'] = portable_data['target_id'].apply(
    lambda x: f"P{x}" if x >= 0 else "P-1"
)

all_clustered = pd.concat([tower_data, portable_data])
all_clustered = all_clustered.sort_index()

In [ ]:
# ============================================================================
# STEP 3: Identify shared targets between DIMMs
# ============================================================================

print("\n" + "="*80)
print("IDENTIFYING SHARED TARGETS")
print("="*80)
print(f"Matching tolerance: RA ≤ {SHARED_TARGET_RA_TOLERANCE}°, DECL ≤ {SHARED_TARGET_DECL_TOLERANCE}°")

def find_shared_targets(tower_df, portable_df, 
                       ra_tolerance=SHARED_TARGET_RA_TOLERANCE, 
                       decl_tolerance=SHARED_TARGET_DECL_TOLERANCE):
    """
    Find targets observed by both DIMMs based on RA/DECL matching
    
    Parameters:
    -----------
    tower_df : DataFrame
        Clustered data from Tower DIMM
    portable_df : DataFrame
        Clustered data from Portable DIMM
    ra_tolerance : float
        Maximum RA difference to consider same target (degrees)
    decl_tolerance : float
        Maximum DECL difference to consider same target (degrees)
    
    Returns:
    --------
    DataFrame with matched targets
    """
    # Get target summaries for each hardware (only valid clusters)
    tower_targets = tower_df[tower_df['target_id'] >= 0].groupby('target_id').agg({
        'ra': 'mean',
        'decl': 'mean',
    }).reset_index()
    tower_targets.columns = ['tower_target_id', 'tower_ra', 'tower_decl']
    
    portable_targets = portable_df[portable_df['target_id'] >= 0].groupby('target_id').agg({
        'ra': 'mean',
        'decl': 'mean',
    }).reset_index()
    portable_targets.columns = ['portable_target_id', 'portable_ra', 'portable_decl']
    
    # Find matches
    matches = []
    
    for _, tower_row in tower_targets.iterrows():
        for _, portable_row in portable_targets.iterrows():
            ra_diff = abs(tower_row['tower_ra'] - portable_row['portable_ra'])
            decl_diff = abs(tower_row['tower_decl'] - portable_row['portable_decl'])
            
            # Handle RA wraparound at 0/360 degrees
            if ra_diff > 180:
                ra_diff = 360 - ra_diff
            
            if ra_diff <= ra_tolerance and decl_diff <= decl_tolerance:
                matches.append({
                    'tower_target_id': tower_row['tower_target_id'],
                    'portable_target_id': portable_row['portable_target_id'],
                    'mean_ra': (tower_row['tower_ra'] + portable_row['portable_ra']) / 2,
                    'mean_decl': (tower_row['tower_decl'] + portable_row['portable_decl']) / 2,
                    'ra_diff': ra_diff,
                    'decl_diff': decl_diff,
                    'tower_ra': tower_row['tower_ra'],
                    'tower_decl': tower_row['tower_decl'],
                    'portable_ra': portable_row['portable_ra'],
                    'portable_decl': portable_row['portable_decl']
                })
    
    return pd.DataFrame(matches)

shared_targets = find_shared_targets(tower_data, portable_data)

print(f"\nTotal shared targets found: {len(shared_targets)}")

if len(shared_targets) > 0:
    print(f"\nAverage coordinate difference for shared targets:")
    print(f"  RA difference: {shared_targets['ra_diff'].mean():.6f}° (±{shared_targets['ra_diff'].std():.6f}°)")
    print(f"  DECL difference: {shared_targets['decl_diff'].mean():.6f}° (±{shared_targets['decl_diff'].std():.6f}°)")

In [ ]:
# ============================================================================
# STEP 4: Analyze temporal overlap for shared targets
# ============================================================================

print("\n" + "="*80)
print("TEMPORAL OVERLAP ANALYSIS")
print("="*80)
print(f"Time window for simultaneous observations: ±{SIMULTANEOUS_TIME_WINDOW_MINUTES} minutes")


def analyze_temporal_overlap(tower_df, portable_df, shared_df, 
                            time_window_minutes=SIMULTANEOUS_TIME_WINDOW_MINUTES):
    """
    Analyze when both DIMMs observe the same target simultaneously
    
    Parameters:
    -----------
    tower_df : DataFrame
        Clustered data from Tower DIMM
    portable_df : DataFrame
        Clustered data from Portable DIMM
    shared_df : DataFrame
        DataFrame of shared targets
    time_window_minutes : float
        Time window to consider observations "simultaneous" (minutes)
    
    Returns:
    --------
    DataFrame with overlap information
    """
    overlap_results = []
    
    for _, match in shared_df.iterrows():
        tower_id = match['tower_target_id']
        portable_id = match['portable_target_id']
        
        # Get observations for this target from each DIMM
        tower_obs = tower_df[tower_df['target_id'] == tower_id].sort_index()
        portable_obs = portable_df[portable_df['target_id'] == portable_id].sort_index()
        
        if len(tower_obs) == 0 or len(portable_obs) == 0:
            continue
        
        # Time ranges
        tower_start = tower_obs.index.min()
        tower_end = tower_obs.index.max()
        portable_start = portable_obs.index.min()
        portable_end = portable_obs.index.max()
        
        # Check for temporal overlap
        overlap_start = max(tower_start, portable_start)
        overlap_end = min(tower_end, portable_end)
        
        has_overlap = overlap_start <= overlap_end
        
        if has_overlap:
            overlap_duration = (overlap_end - overlap_start).total_seconds() / 60  # minutes
            
            # Count simultaneous observations (within time window)
            simultaneous_count = 0
            time_delta = timedelta(minutes=time_window_minutes)
            
            for t_time in tower_obs.index:
                # Check if any portable observation is within time window
                time_matches = portable_obs[
                    (portable_obs.index >= t_time - time_delta) &
                    (portable_obs.index <= t_time + time_delta)
                ]
                if len(time_matches) > 0:
                    simultaneous_count += 1
        else:
            overlap_duration = 0
            simultaneous_count = 0
        
        overlap_results.append({
            'tower_target_id': tower_id,
            'portable_target_id': portable_id,
            'mean_ra': match['mean_ra'],
            'mean_decl': match['mean_decl'],
            'tower_observations': len(tower_obs),
            'portable_observations': len(portable_obs),
            'tower_start': tower_start,
            'tower_end': tower_end,
            'tower_duration_min': (tower_end - tower_start).total_seconds() / 60,
            'portable_start': portable_start,
            'portable_end': portable_end,
            'portable_duration_min': (portable_end - portable_start).total_seconds() / 60,
            'has_temporal_overlap': has_overlap,
            'overlap_start': overlap_start if has_overlap else None,
            'overlap_end': overlap_end if has_overlap else None,
            'overlap_duration_min': overlap_duration,
            'simultaneous_obs_count': simultaneous_count
        })
    
    return pd.DataFrame(overlap_results)

overlap_analysis = analyze_temporal_overlap(tower_data, portable_data, shared_targets)

# Summary statistics
if len(overlap_analysis) > 0:
    n_with_overlap = overlap_analysis['has_temporal_overlap'].sum()
    n_without_overlap = len(overlap_analysis) - n_with_overlap
    
    print(f"\nShared targets with temporal overlap: {n_with_overlap}")
    print(f"Shared targets without temporal overlap: {n_without_overlap}")
    
    if n_with_overlap > 0:
        overlap_subset = overlap_analysis[overlap_analysis['has_temporal_overlap']]
        print(f"\nFor targets with overlap:")
        print(f"  Average overlap duration: {overlap_subset['overlap_duration_min'].mean():.1f} minutes")
        print(f"  Maximum overlap duration: {overlap_subset['overlap_duration_min'].max():.1f} minutes")
        print(f"  Average simultaneous observations: {overlap_subset['simultaneous_obs_count'].mean():.1f}")


# ============================================================================
# STEP 5: Generate summary statistics
# ============================================================================

print("\n" + "="*80)
print("GENERATING SUMMARY STATISTICS")
print("="*80)

def generate_target_summary(data, hardware_name):
    """Generate summary statistics for targets"""
    valid_data = data[data['target_id'] >= 0]
    
    if len(valid_data) == 0:
        return pd.DataFrame()
    
    summary = valid_data.groupby('target_id').agg({
        'ra': ['mean', 'std', 'count'],
        'decl': ['mean', 'std']
    }).round(6)
    
    summary.columns = ['RA_mean', 'RA_std', 'Count', 'DECL_mean', 'DECL_std']
    
    # Add time information
    time_info = valid_data.groupby('target_id').apply(
        lambda x: pd.Series({
            'start_time': x.index.min(),
            'end_time': x.index.max(),
            'duration_minutes': (x.index.max() - x.index.min()).total_seconds() / 60
        }), include_groups=False
    )
    
    summary = summary.join(time_info)
    summary['hardware'] = hardware_name
    
    return summary

# Generate summaries
tower_summary = generate_target_summary(tower_data, 'Tower DIMM')
portable_summary = generate_target_summary(portable_data, 'Portable DIMM')

print(f"\nTower DIMM summary: {len(tower_summary)} significant targets")
print(f"Portable DIMM summary: {len(portable_summary)} significant targets")

In [ ]:
# ============================================================================
# STEP 6: Create visualizations
# ============================================================================

print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. RA vs DECL for Tower DIMM
ax1 = fig.add_subplot(gs[0, 0])
tower_valid = tower_data[tower_data['target_id'] >= 0]
if len(tower_valid) > 0:
    scatter1 = ax1.scatter(tower_valid['ra'], tower_valid['decl'],
                          c=tower_valid['target_id'], cmap='tab20',
                          alpha=0.6, s=10)
    ax1.set_xlabel('Right Ascension (degrees)', fontsize=10)
    ax1.set_ylabel('Declination (degrees)', fontsize=10)
    ax1.set_title(f'Tower DIMM Targets\n({len(tower_summary)} significant targets)', 
                  fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    if len(tower_summary) <= 20:
        plt.colorbar(scatter1, ax=ax1, label='Target ID')

# 2. RA vs DECL for Portable DIMM
ax2 = fig.add_subplot(gs[0, 1])
portable_valid = portable_data[portable_data['target_id'] >= 0]
if len(portable_valid) > 0:
    scatter2 = ax2.scatter(portable_valid['ra'], portable_valid['decl'],
                          c=portable_valid['target_id'], cmap='tab20',
                          alpha=0.6, s=10)
    ax2.set_xlabel('Right Ascension (degrees)', fontsize=10)
    ax2.set_ylabel('Declination (degrees)', fontsize=10)
    ax2.set_title(f'Portable DIMM Targets\n({len(portable_summary)} significant targets)', 
                  fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    if len(portable_summary) <= 20:
        plt.colorbar(scatter2, ax=ax2, label='Target ID')

# 3. Shared targets overlay
ax3 = fig.add_subplot(gs[0, 2])
if len(tower_valid) > 0:
    ax3.scatter(tower_valid['ra'], tower_valid['decl'],
               c='blue', alpha=0.3, s=5, label='Tower DIMM')
if len(portable_valid) > 0:
    ax3.scatter(portable_valid['ra'], portable_valid['decl'],
               c='red', alpha=0.3, s=5, label='Portable DIMM')
if len(shared_targets) > 0:
    ax3.scatter(shared_targets['mean_ra'], shared_targets['mean_decl'],
               c='green', marker='*', s=200, edgecolors='black',
               linewidths=1, label=f'Shared ({len(shared_targets)})', zorder=5)
ax3.set_xlabel('Right Ascension (degrees)', fontsize=10)
ax3.set_ylabel('Declination (degrees)', fontsize=10)
ax3.set_title('Shared Targets Overlay', fontsize=12, fontweight='bold')
ax3.legend(loc='best', fontsize=8)
ax3.grid(True, alpha=0.3)

# 4. Observations per target - Tower DIMM
ax4 = fig.add_subplot(gs[1, 0])
if len(tower_summary) > 0:
    tower_counts = tower_summary['Count'].sort_values(ascending=False)[:30]
    ax4.barh(range(len(tower_counts)), tower_counts.values, color='steelblue', alpha=0.7)
    ax4.set_yticks(range(len(tower_counts)))
    ax4.set_yticklabels([f"T{idx}" for idx in tower_counts.index], fontsize=8)
    ax4.set_xlabel('Number of Observations', fontsize=10)
    ax4.set_title(f'Tower DIMM: Top {min(30, len(tower_counts))} Targets', 
                  fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='x')
    ax4.invert_yaxis()

# 5. Observations per target - Portable DIMM
ax5 = fig.add_subplot(gs[1, 1])
if len(portable_summary) > 0:
    portable_counts = portable_summary['Count'].sort_values(ascending=False)[:30]
    ax5.barh(range(len(portable_counts)), portable_counts.values, color='coral', alpha=0.7)
    ax5.set_yticks(range(len(portable_counts)))
    ax5.set_yticklabels([f"P{idx}" for idx in portable_counts.index], fontsize=8)
    ax5.set_xlabel('Number of Observations', fontsize=10)
    ax5.set_title(f'Portable DIMM: Top {min(30, len(portable_counts))} Targets', 
                  fontsize=12, fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='x')
    ax5.invert_yaxis()

# 6. Temporal overlap visualization
ax6 = fig.add_subplot(gs[1, 2])
if len(overlap_analysis) > 0 and overlap_analysis['has_temporal_overlap'].any():
    overlap_subset = overlap_analysis[overlap_analysis['has_temporal_overlap']].head(20)
    y_pos = range(len(overlap_subset))
    
    for i, (_, row) in enumerate(overlap_subset.iterrows()):
        ax6.plot([row['tower_start'], row['tower_end']], [i, i],
                'b-', linewidth=6, alpha=0.5, label='Tower' if i == 0 else '')
        ax6.plot([row['portable_start'], row['portable_end']], [i+0.2, i+0.2],
                'r-', linewidth=6, alpha=0.5, label='Portable' if i == 0 else '')
        if row['overlap_start'] is not None:
            ax6.plot([row['overlap_start'], row['overlap_end']], [i+0.1, i+0.1],
                    'g-', linewidth=8, alpha=0.8, label='Overlap' if i == 0 else '')
    
    ax6.set_yticks(y_pos)
    ax6.set_yticklabels([f"T{int(row['tower_target_id'])}-P{int(row['portable_target_id'])}"
                         for _, row in overlap_subset.iterrows()], fontsize=8)
    ax6.set_xlabel('Time', fontsize=10)
    ax6.set_title('Temporal Overlap (Top 20)', fontsize=12, fontweight='bold')
    ax6.legend(loc='best', fontsize=8)
    ax6.grid(True, alpha=0.3, axis='x')
    fig.autofmt_xdate()

# 7. Duration comparison
ax7 = fig.add_subplot(gs[2, 0])
if len(tower_summary) > 0 and len(portable_summary) > 0:
    ax7.hist(tower_summary['duration_minutes'], bins=30, alpha=0.6,
            label='Tower DIMM', color='blue')
    ax7.hist(portable_summary['duration_minutes'], bins=30, alpha=0.6,
            label='Portable DIMM', color='red')
    ax7.set_xlabel('Duration (minutes)', fontsize=10)
    ax7.set_ylabel('Number of Targets', fontsize=10)
    ax7.set_title('Target Duration Distribution', fontsize=12, fontweight='bold')
    ax7.legend()
    ax7.grid(True, alpha=0.3, axis='y')
    ax7.set_yscale('log')

# 8. Coordinate difference for shared targets
ax8 = fig.add_subplot(gs[2, 1])
if len(shared_targets) > 0:
    ax8.scatter(shared_targets['ra_diff'], shared_targets['decl_diff'],
               alpha=0.6, s=50, c='purple')
    ax8.set_xlabel('RA Difference (degrees)', fontsize=10)
    ax8.set_ylabel('DECL Difference (degrees)', fontsize=10)
    ax8.set_title('Coordinate Differences\nfor Shared Targets', fontsize=12, fontweight='bold')
    ax8.grid(True, alpha=0.3)
    ax8.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax8.axvline(x=0, color='k', linestyle='--', alpha=0.3)

# 9. Summary statistics table
ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')

summary_text = "ANALYSIS SUMMARY v2.0\n" + "="*35 + "\n\n"
summary_text += f"PARAMETERS:\n"
summary_text += f"  eps={CLUSTERING_EPS}°, min_samples={CLUSTERING_MIN_SAMPLES}\n"
summary_text += f"  Filter: ≥{MIN_OBSERVATIONS} obs OR ≥{MIN_DURATION_MINUTES}min\n\n"
summary_text += f"TOWER DIMM:\n"
summary_text += f"  Significant targets: {len(tower_summary)}\n"
if len(tower_summary) > 0:
    summary_text += f"  Total obs: {tower_summary['Count'].sum():,.0f}\n"
    summary_text += f"  Filtered: {tower_stats['filtered_clusters']} clusters\n\n"
summary_text += f"PORTABLE DIMM:\n"
summary_text += f"  Significant targets: {len(portable_summary)}\n"
if len(portable_summary) > 0:
    summary_text += f"  Total obs: {portable_summary['Count'].sum():,.0f}\n"
    summary_text += f"  Filtered: {portable_stats['filtered_clusters']} clusters\n\n"
summary_text += f"SHARED TARGETS: {len(shared_targets)}\n"
if len(overlap_analysis) > 0:
    n_overlap = overlap_analysis['has_temporal_overlap'].sum()
    summary_text += f"  With overlap: {n_overlap}\n"
    summary_text += f"  Without overlap: {len(overlap_analysis) - n_overlap}\n"

ax9.text(0.1, 0.9, summary_text, transform=ax9.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.savefig('multi_dimm_analysis_v2.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved: multi_dimm_analysis_v2.png")